In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded ✓")

In [ ]:
# Load the dataset
df = pd.read_csv('../data/bronze/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv', low_memory=False)

print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
# now, I will check 
df

In [ ]:
# Target variable distribution, 
# I will check the distribution of the target variable 'loan_status' 
# to understand the balance of the classes.
print(df['loan_status'].value_counts())


In [ ]:
#after a analysis of the target variable, i see the cases that i want to focus on are 
#charged off, default, and late (31-120 days)
#  because they represent the cases where the borrower has not fulfilled their loan obligations.
# so i make a first step of filtering the dataset to include only these relevant cases
# along with the fully paid cases for comparison.
#for a better comppresion general i will create a new column 'default_flag'
#  that will be 1 for the default cases and 0 for the fully paid cases, 
# this will help me in the analysis and modeling steps later on.
default_status = ['Charged Off', 'Late (31-120 days)', 'Default']
paid_status = ['Fully Paid']

# Filter only completed loans
df_clean = df[df['loan_status'].isin(default_status + paid_status)].copy()

# Create default flag
df_clean['default_flag'] = df_clean['loan_status'].isin(default_status).astype(int)

print(f"Dataset size after filter: {df_clean.shape[0]:,}")
print(f"\nDefault flag distribution:")
print(df_clean['default_flag'].value_counts())
print(f"\nDefault rate: {df_clean['default_flag'].mean()*100:.2f}%")

In [ ]:
# Null analysis
null_pct = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].round(2))


In [ ]:
# I will check for missing values in the dataset to understand the extent of data cleaning required.
#in this step, I will analyze the percentage of missing values in each column to identify 
# which features may need to be dropped or imputed.
# Classify columns by null percentage
print("Columns with > 50% nulls:", (null_pct > 50).sum())
print("Columns with 20-50% nulls:", ((null_pct >= 20) & (null_pct <= 50)).sum())
print("Columns with < 20% nulls:", (null_pct < 20).sum())

## 2. Data Cleaning

### Column Selection
We dropped columns based on three criteria:
- **High null percentage (>50%)**: Not enough data to be useful
- **Redundant columns**: Captured by other variables (e.g. `grade` → `sub_grade`)
- **No analytical value**: Identifiers, URLs, free text (e.g. `id`, `url`, `desc`)

### Null Imputation Strategy
- `mths_since_recent_inq` / `mths_since_recent_bc`: Converted to categories (Last Year / 1-3 Years Ago / More than 3 Years / No Record)
- `emp_title`: Filled with "Unknown"
- `emp_length`: Filled with 0 (assumed unemployed/not reported)
- `bc_util`, `percent_bc_gt_75`, `bc_open_to_buy`: Filled with 0 (no credit cards)
- Remaining nulls (<5%): Dropped rows — dataset large enough to absorb the loss

In [ ]:
# second step of the data cleaning process will be to drop columns that are either highly incomplete or not relevant for the analysis.

# ── Column Selection ──────────────────────────────────────────────
# Drop high null columns (>50%)
high_null_cols = null_pct[null_pct > 50].index.tolist()

# Drop redundant or irrelevant columns
cols_to_drop = list(set(high_null_cols + [
    # Redundant with funded_amnt
    'loan_amnt',
    # Redundant funding columns (investor perspective)
    'funded_amnt_inv', 'out_prncp_inv', 'total_pymnt_inv',
    # Redundant risk columns
    'grade', 'fico_range_high', 'last_fico_range_high',
    'num_sats', 'num_bc_tl', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
    'pub_rec_bankruptcies', 'tax_liens',
    # Payment components (captured in total_pymnt)
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    # Post-default columns
    'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
    # Time-since columns (not intuitive for BI)
    'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op',
    'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
    # No analytical value
    'url', 'desc', 'title', 'zip_code', 'id',
    'pymnt_plan', 'initial_list_status', 'policy_code',
    'hardship_flag', 'disbursement_method', 'debt_settlement_flag',
    # Redundant open accounts
    'open_acc',
    # Redundant num_* columns
    'num_actv_bc_tl', 'num_bc_sats', 'num_il_tl',
    'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0',
    'num_tl_op_past_12m',
]))

df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')
print(f"Final shape: {df_clean.shape}")
print(df_clean.columns.tolist())

In [ ]:
# an extra step of cleaning, I will drop some additional columns that are redundant or
#  have been replaced by more comprehensive features.
extra_drop   = [
    'tot_cur_bal',
    'total_bc_limit',
    'total_il_high_credit_limit',
    'tot_hi_cred_lim',
]

df_clean = df_clean.drop(columns=extra_drop_5, errors='ignore')
print(f"Shape: {df_clean.shape}")
print(df_clean.columns.tolist())


In [ ]:
df_clean

In [ ]:
print(df_clean.dtypes)

In [ ]:
# ── Feature Transformation ──────────────────────────────────────────────

# 1. term: "36 months" → 36
df_clean['term'] = df_clean['term'].str.extract('(\d+)').astype(int)

# 2. emp_length: "10+ years" → 10, "< 1 year" → 0
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}
df_clean['emp_length'] = df_clean['emp_length'].map(emp_map)

# 3. issue_d: "Dec-2015" → datetime
df_clean['issue_d'] = pd.to_datetime(df_clean['issue_d'], format='%b-%Y')

# 4. earliest_cr_line → credit history in years
df_clean['earliest_cr_line'] = pd.to_datetime(df_clean['earliest_cr_line'], format='%b-%Y')
df_clean['credit_history_years'] = (df_clean['issue_d'] - df_clean['earliest_cr_line']).dt.days / 365
df_clean = df_clean.drop(columns=['earliest_cr_line'])

print("Transformations done ✓")
print(df_clean[['term', 'emp_length', 'issue_d', 'credit_history_years']].head())

In [ ]:
null_remaining = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)
print(null_remaining[null_remaining > 0].round(2))

In [ ]:
# ── Null Imputation ──────────────────────────────────────────────

# 1. mths_since_recent_inq → binning
def bin_mths_inq(x):
    if pd.isna(x): return 'No Record'
    elif x <= 12: return 'Last Year'
    elif x <= 36: return '1-3 Years Ago'
    else: return 'More than 3 Years'

df_clean['mths_since_recent_inq_cat'] = df_clean['mths_since_recent_inq'].apply(bin_mths_inq)
df_clean = df_clean.drop(columns=['mths_since_recent_inq'])

# 2. mths_since_recent_bc → binning
def bin_mths_bc(x):
    if pd.isna(x): return 'No Record'
    elif x <= 12: return 'Last Year'
    elif x <= 36: return '1-3 Years Ago'
    else: return 'More than 3 Years'

df_clean['mths_since_recent_bc_cat'] = df_clean['mths_since_recent_bc'].apply(bin_mths_bc)
df_clean = df_clean.drop(columns=['mths_since_recent_bc'])

# 3. emp_title → Unknown
df_clean['emp_title'] = df_clean['emp_title'].fillna('Unknown')

# 4. emp_length → 0
df_clean['emp_length'] = df_clean['emp_length'].fillna(0)

# 5. bc columns → 0
df_clean[['bc_util', 'percent_bc_gt_75', 'bc_open_to_buy']] = df_clean[['bc_util', 'percent_bc_gt_75', 'bc_open_to_buy']].fillna(0)

# 6. Drop remaining nulls
df_clean = df_clean.dropna()

print(f"Shape after imputation: {df_clean.shape}")
print(f"Remaining nulls: {df_clean.isnull().sum().sum()}")

In [ ]:
# Save to silver layer
df_clean.to_csv('../data/silver/loans_clean.csv', index=False)
print("Saved to silver ✓")